<a href="https://colab.research.google.com/github/balajiduddukuri/Langchain_practice/blob/Ultimate-Content-Repurposer/langchain_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
#practice programs

In [9]:

# ===============================
# 1) Install dependencies
# ===============================
!pip install -q langchain langchain-google-genai google-generativeai


In [10]:

# Optional utilities if you plan to extend to RAG later:
# !pip install -q chromadb faiss-cpu pypdf tiktoken docarray

# ===============================
# 2) Imports & API key setup
# ===============================
import os
from google.colab import userdata

# LangChain core + Google Gemini Chat wrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate

# Load and assert API key (store it in Colab with key name: "google_api_key")
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
assert os.environ.get("GOOGLE_API_KEY"), (
    "Missing GOOGLE_API_KEY. In Colab, run: "
    "from google.colab import userdata; userdata.set('google_api_key', 'YOUR_KEY')"
)
print("Google API key is loaded successfully.")

Google API key is loaded successfully.


In [11]:

# ===============================
# 3) Helper: build Gemini LLM
# ===============================
def get_llm(model: str = "gemini-3-flash-preview", temperature: float = 0):
    """
    Returns a LangChain LLM wrapper for Google Gemini chat models.
    Common models:
      - gemini-1.5-flash : fast, cheaper, good for most chat tasks
      - gemini-1.5-pro   : higher quality, reasoning
    """
    return ChatGoogleGenerativeAI(model=model, temperature=temperature)

In [12]:
import os
import requests

API_KEY = os.getenv("GOOGLE_API_KEY") # Corrected: Changed to GOOGLE_API_KEY
if not API_KEY:
    raise SystemExit("Set GOOGLE_API_KEY in your environment.") # Changed message for consistency

# Primary endpoint (v1). Some accounts/regions might still expose models under v1beta.
BASES = [
    "https://generativelanguage.googleapis.com/v1/models",
    "https://generativelanguage.googleapis.com/v1beta/models",  # fallback
]

def list_gemini_models() -> list[str]:
    for base in BASES:
        try:
            r = requests.get(base, params={"key": API_KEY}, timeout=20)
            r.raise_for_status()
            data = r.json()
            models = data.get("models", [])
            names = []
            for m in models:
                # Example: "name": "models/gemini-1.5-flash"
                n = m.get("name", "")
                if n.startswith("models/"):
                    n = n.split("/", 1)[1]
                if n:
                    names.append(n)
            if names:
                return sorted(set(names))
        except requests.HTTPError as e:
            # try the next base; if none works, rethrow on last iteration
            if base is BASES[-1]:
                raise
            continue
    return []

In [13]:
#if __name__ == "__main__":
names = list_gemini_models()
print("\n=== Gemini models (live) ===")
if not names:
   print("(No models returned. Check your API key or permissions.)")
for n in names:
   print(" -", n)


=== Gemini models (live) ===
 - gemini-2.0-flash
 - gemini-2.0-flash-001
 - gemini-2.0-flash-lite
 - gemini-2.0-flash-lite-001
 - gemini-2.5-flash
 - gemini-2.5-flash-lite
 - gemini-2.5-pro


In [17]:

# ===============================
# 4) Run the same query with two Gemini models
# ===============================
query = "Explain the uses of LangChain Framework in bullet points"

for model in ["gemini-3-flash-preview", "gemini-3-flash-preview"]:
    llm = get_llm(model=model, temperature=0)
    response = llm.invoke([HumanMessage(content=query)])
    print(f"\n--- {model} ---\n{response.content}")

from markdown import markdown

#text = """YOUR_LONG_TEXT_HERE"""

html = markdown(response.text, extensions=["extra"])  # 'extra' handles tables, fenced code, etc.
#print(html)
# Display formatted HTML in Colab
from IPython.display import display, HTML
display(HTML(html))


--- gemini-3-flash-preview ---
[{'type': 'text', 'text': '**LangChain** is an open-source framework designed to simplify the creation of applications using Large Language Models (LLMs). It acts as a "glue" that connects LLMs to external data sources, APIs, and logic.\n\nHere are the primary uses of the LangChain framework:\n\n### 1. Retrieval Augmented Generation (RAG)\n*   **Connecting to Private Data:** LangChain allows LLMs to access and "read" your private documents (PDFs, emails, databases) without needing to retrain the model.\n*   **Knowledge Bases:** It is widely used to build internal company wikis or customer support bots that answer questions based on specific, up-to-date documentation.\n\n### 2. Building Conversational AI (Chatbots)\n*   **Memory Management:** Standard LLMs are "stateless" (they don\'t remember previous messages). LangChain provides utilities to give bots "memory," allowing them to maintain context over long conversations.\n*   **Persona Consistency:** It 

In [19]:
# ===============================
# 5) PromptTemplate demo (unchanged)
# ===============================
template = """
You are an expert AI tutor.

Explain {topic} in simple terms.
Give examples.
Audience: {audience}
"""

prompt = PromptTemplate(
    input_variables=["topic", "audience"],
    template=template
)

formatted_prompt = prompt.format(
    topic="GAN",
    audience="Beginner AI Students"
)

print("\n--- Formatted Prompt ---\n", formatted_prompt)

llm = get_llm("gemini-3-flash-preview", temperature=0)
resp = llm.invoke(formatted_prompt)  # You can pass a plain string to invoke
print("\n--- Model Output ---\n", resp.content)


--- Formatted Prompt ---
 
You are an expert AI tutor.

Explain GAN in simple terms.
Give examples.
Audience: Beginner AI Students


--- Model Output ---
 [{'type': 'text', 'text': 'Hello! As your AI tutor, I’m excited to help you understand one of the most fascinating concepts in modern Artificial Intelligence: **Generative Adversarial Networks**, or **GANs**.\n\nIn short, a GAN is a way to train a computer to **create things** (like images, music, or text) that look like they were made by humans.\n\n---\n\n### 1. The Simple Analogy: The Art Forger and the Detective\n\nThe best way to understand a GAN is to imagine a game between two people:\n\n1.  **The Generator (The Art Forger):** This person’s goal is to create a fake painting that looks exactly like a masterpiece (e.g., a Van Gogh). At first, they are terrible at it—they might just scribble with a crayon.\n2.  **The Discriminator (The Art Detective):** This person’s goal is to look at a painting and decide if it is a **Real** ma

In [22]:
# pip install markdown
from markdown import markdown

text = """YOUR_LONG_TEXT_HERE"""

html = markdown(resp.text, extensions=["extra"])  # 'extra' handles tables, fenced code, etc.
#print(html)
display(HTML(html))

In [23]:
#SARVESH

from langchain_core.prompts import PromptTemplate

template = """
You are an expert AI tutor.

Explain {topic} in simple terms.
Give examples.
Audience: {audience}
"""

prompt = PromptTemplate(
    input_variables = ['topic', 'audience'],
    template = template
)

formatted_prompt = prompt.format(
    topic = "Photosynthasis",
    audience = "Advanced Science AI Students"
)

print(formatted_prompt)



You are an expert AI tutor.

Explain Photosynthasis in simple terms.
Give examples.
Audience: Advanced Science AI Students



In [26]:
llm = get_llm("gemini-3-flash-preview", temperature=0)
resp = llm.invoke(formatted_prompt)  # You can pass a plain string to invoke
print("\n--- Model Output ---\n", resp.content)


--- Model Output ---
 [{'type': 'text', 'text': 'Welcome, class. As advanced AI students, you are used to thinking about **energy optimization, signal transduction, and complex system architectures.**\n\nTo understand photosynthesis, stop thinking of it as a "plant thing" and start thinking of it as a **biological solar-to-chemical transducer.** It is a highly evolved, multi-stage algorithm designed to solve the problem of energy storage.\n\nHere is photosynthesis explained through the lens of systems engineering and bio-computation.\n\n---\n\n### 1. The High-Level Architecture (The Equation)\nIn AI terms, photosynthesis is a **generative process** that takes low-energy, high-entropy inputs and organizes them into high-energy, low-entropy outputs.\n\n**The Input/Output Mapping:**\n$$6CO_2 + 6H_2O + \\text{Photons} \\rightarrow C_6H_{12}O_6 + 6O_2$$\n\n*   **Inputs:** Carbon Dioxide (Data), Water (Hardware support), and Photons (The Power Supply).\n*   **Output:** Glucose (The stored "

In [27]:
html = markdown(resp.text, extensions=["extra"])  # 'extra' handles tables, fenced code, etc.
#print(html)
display(HTML(html))